# GOLD ATP HEAD-TO-HEAD

## Imports

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.window import Window
import pandas as pd

import os
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from dotenv import load_dotenv
load_dotenv()

True

## Init spark

In [3]:
try:
    spark = SparkSession.builder.appName("fact_head_to_head").getOrCreate()
except Exception as e:
    print(e)

In [4]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 200)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 50)

## Load database

In [ ]:
# silver
tb_player_match = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "silver.tb_atp_tournaments")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
    )

# gold
tb_tournaments = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "gold.dim_tournaments")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
    )

tb_players = (
    spark.read
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "gold.dim_players")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .load()
    )

## Head-To-Head

In [6]:
df = (
    tb_player_match.alias("p_m")
    .join(tb_tournaments.alias("t"), "TOURNEY_ID", 'left')
    .join(
        tb_players.alias("p_w"),
        (f.col("p_m.PLAYER_ID").cast("string") == f.col("p_w.PLAYER_ID").cast("string"))
        | (
            f.col("p_m.PLAYER_ID").cast("string")
            == f.col("p_w.PLAYER_ID_OLD").cast("string")
        ),
        "left",
    )
    .join(
        tb_players.alias("p_l"),
        (f.col("p_m.PLAYER_OPPONENT_ID").cast("string") == f.col("p_l.PLAYER_ID").cast("string"))
        | (
            f.col("p_m.PLAYER_OPPONENT_ID").cast("string")
            == f.col("p_l.PLAYER_ID_OLD").cast("string")
        ),
        "left",
    )

    .groupBy("p_w.SK_PLAYER", "p_l.SK_PLAYER")
    .agg(
        f.count("p_m.MATCH_ID").alias("TOTAL_MATCHES"),

        f.sum(f.when(f.col("PLAYER_IS_WINNER") == True, f.lit(1)).otherwise(f.lit(0))).alias("TOTAL_WINS"),
        f.sum(f.when(f.col("PLAYER_IS_WINNER") == False, f.lit(1)).otherwise(f.lit(0))).alias("TOTAL_LOSSES"),
    )
    .select(
        f.col("p_w.SK_PLAYER").alias("SK_PLAYER"),
        f.col("p_l.SK_PLAYER").alias("SK_PLAYER_OPPONENT"),
        f.col("TOTAL_MATCHES").alias("TOTAL_MATCHES"),
        f.col("TOTAL_WINS").alias("TOTAL_WINS"),
        f.col("TOTAL_LOSSES").alias("TOTAL_LOSSES")
    )
)

## Save dataframe

### Local

In [9]:
df.toPandas().to_csv(
    r"../../../data/gold/fact/fact_player_head_to_head.csv",
    index=False,
    sep=",",
    encoding="utf-8"
)

### Supabase

In [ ]:
(
df.write
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "gold.fact_player_head_to_head")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .mode("overwrite")
    .save()
)